In [ ]:
!pip install faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 957.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

# Indexing Documents in a Vector Database (FAISS)

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load a pre-trained embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # Open-source

# Example documents to store in the vector database
documents = [
    "Machine learning is a method of data analysis that automates analytical model building.",
    "Artificial intelligence is intelligence demonstrated by machines, as opposed to human intelligence.",
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "Vector databases allow for efficient similarity search in high-dimensional spaces.",
    "Natural language processing (NLP) enables machines to understand and generate human language.",
]

# Generate embeddings for documents
document_embeddings = embedding_model.encode(documents)

# Convert to NumPy array for FAISS
document_embeddings = np.array(document_embeddings).astype("float32")

# Initialize FAISS index (IndexFlatL2 for standard cosine similarity search)
index = faiss.IndexFlatL2(document_embeddings.shape[1])  # L2 (Euclidean) distance index
index.add(document_embeddings)  # Add embeddings to FAISS index

print(f"Indexed {index.ntotal} documents in the vector database.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 5 documents in the vector database.


# Querying the Vector Database


In [ ]:
def search_faiss(query, top_k=2):
    # Convert query to an embedding
    query_embedding = embedding_model.encode([query]).astype("float32")

    # Search in FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # Retrieve matching documents
    results = [(documents[idx], distances[0][i]) for i, idx in enumerate(indices[0])]

    return results

# Example Query
query = "What is deep learning?"
results = search_faiss(query, top_k=2)

# Display results
for doc, score in results:
    print(f"Retrieved: '{doc}' (Score: {score:.4f})")


Retrieved: 'Deep learning is a subset of machine learning that uses neural networks with many layers.' (Score: 0.3445)
Retrieved: 'Machine learning is a method of data analysis that automates analytical model building.' (Score: 1.1146)


# Using Retrieved Documents for Retrieval-Augmented Generation (RAG)

In [ ]:
from transformers import pipeline

# Load open-source GPT model
generator = pipeline("text-generation", model="EleutherAI/gpt-neo-1.3B")

def generate_with_rag(query):
    # Retrieve relevant context
    retrieved_docs = search_faiss(query, top_k=2)
    context = " ".join([doc for doc, _ in retrieved_docs])  # Concatenate docs

    # Construct prompt
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    # Generate response
    output = generator(prompt, max_length=100, do_sample=True)

    return output[0]["generated_text"]

# Example usage
query = "Explain deep learning."
response = generate_with_rag(query)
print(response)


config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Context: Deep learning is a subset of machine learning that uses neural networks with many layers. Machine learning is a method of data analysis that automates analytical model building.

Question: Explain deep learning.

Answer: It is a set of techniques used to learn how to detect and predict data. In the context of machine learning, the deep is in the middle between shallow and complex. The deep layers use deep learning techniques to build an analytical model. To begin, you should know the theory
